In [13]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from q3dfit.q3dout import load_q3dout, q3dout
from q3dfit.q3dutil import get_q3dio, get_spaxels
from q3dfit.readcube import Cube

In [28]:
filename_q3di = "input/q3di1.npy"
q3di = get_q3dio(filename_q3di)
cube = q3di.load_cube()
print(cube.dat.shape)
print(cube.err.shape)
data = cube.dat
var = cube.var
print(np.nanmin(var), np.nanmax(var))
nspax, colarr, rowarr = get_spaxels(cube)
for ispax in range(0, nspax):
    i = colarr[ispax]
    j = rowarr[ispax]
    try:
        q3do: q3dout = load_q3dout(q3di, i+1, j+1) 
    except FileNotFoundError:
        continue
    if q3do.docontfit:
        data[i, j, q3do.fitran_indx] -= q3do.cont_fit

with fits.open("4C1971_with_data_quality.fits") as hdul:
    hdul.info()
    hdul[1].data = data.T * cube.fluxnorm
    hdul[1].header["BUNIT"] = "erg/s/cm2/micron"
    hdul[2].data = var.T * cube.fluxnorm**2
    hdul[2].header["BUNIT"] = "erg/s/cm2/micron"
    hdul.writeto("4C1971_wo_continuum.fits", overwrite=True)


Cube: No wavelength units in header; using micron
Size of data cube: [70, 83, 3815]
Wavelength range: [1.66020, 3.17054] micron
Dispersion: 0.00040 micron
Input flux units: MJy/sr
Input wave units: micron
Output flux units: erg/s/cm2/micron
Output wave units: micron
NB: q3dfit uses output units for internal calculations.
(70, 83, 3815)
(70, 83, 3815)
0.12366852236761013 5105.752917171183
Filename: 4C1971_with_data_quality.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     374   ()      
  1  SCI           1 ImageHDU        34   (70, 83, 3815)   float64   
  2  SCI           1 ImageHDU        34   (70, 83, 3815)   float64   
  3                1 ImageHDU         8   (70, 83, 3815)   float64   


In [29]:
with fits.open("4C1971_wo_continuum.fits") as hdul:
    hdul.info()
    # print(repr(hdul[2].header))
    print(hdul[2].data.shape)
    print(np.nanmin(hdul[1].data))
    print(np.nanmax(hdul[1].data))
    print(np.nanmin(hdul[2].data))
    print(np.nanmax(hdul[2].data))

Filename: 4C1971_wo_continuum.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     374   ()      
  1  SCI           1 ImageHDU        34   (70, 83, 3815)   float64   
  2  SCI           1 ImageHDU        34   (70, 83, 3815)   float64   
  3                1 ImageHDU         8   (70, 83, 3815)   float64   
(3815, 83, 70)
-2.0833969227448163e-15
1.8800684509708685e-12
1.2366852236761015e-35
5.105752917171184e-31
